# OTS-SC-DCOPF — Can switching make the full N-1 set feasible?

**The question.** With the full contingency set ($\mathcal{C}_T$ and $\mathcal{C}_G$ complete, no
exclusions), the SC-DCOPF on the original topology is **infeasible**: no base-case dispatch
survives every contingency. Three elements cause it (Section 10 of `../../sc-dcopf/sc-dcopf.ipynb`):
transformer **8-5** (infeasible even alone), line **38-65** and the **bus-10 generator** (joint
conflicts). Analysis B asks: **does some topology (some set of open lines) make the full N-1 set
feasible, and with how few openings ($K_{\min}$)?**

If yes, the OTS does more than reduce cost: it **extends the secure operating region** to a case
the fixed topology cannot handle. If no (for every topology with up to $N$ openings), that is
also a result: switching alone cannot fix it.

**What we already know** (`../RESULTS_NOTES.md`, `../../TODO.md`):
- $K \le 3$: **proven infeasible**.
- Base + 8-5 alone needs 3 openings; base + 8-5 + gen bus 10 needs 4; base + 8-5 + 38-65 is
  feasible with 7 openings (found by a heuristic, not proven minimal).
- Base + 8-5 + 38-65 + gen bus 10 (the "4-scenario core"): **undecided**. Gurobi found no
  topology in 8 minutes, and could not prove that none exists either.

### Why we do not solve the full MILP directly

The full OTS has 197 states × (118 buses + 186 branches) and 177 binaries. Each $z_{ij}$ appears in
every state, so a single open line changes all 197 power flows. Gurobi spends hours without
finding even one feasible topology, and then we learn nothing.

### The idea: solve a small subset of states first (the "core")

Take only some of the states (always including the base case). Call it the **core**. The core
problem has fewer constraints than the full one, so it is a **relaxation**:

| Core result | What it means for the full problem |
|---|---|
| **infeasible** | **infeasible too** (if a few states cannot be satisfied, all of them cannot). This is a proof. |
| **feasible**, topology $z^*$ | maybe. Fix $z^*$ and solve the full problem on it. With $z$ fixed it is an LP (seconds). |

If the full LP on $z^*$ is feasible, we have a topology that works for the full N-1 set. If it is
not, Gurobi tells us **which states conflict** (the IIS, Section 6). Those states are added to the
core and the core is solved again. This loop is called **constraint generation**:

```
core = {base, 8-5, 38-65, gen bus 10}
repeat:
    solve the OTS on the core           -> infeasible?  stop: proven infeasible
                                         -> topology z*
    solve the full LP with z fixed = z* -> feasible?    stop: z* works for the full N-1 set
                                         -> infeasible: add the conflicting states to the core
```

### Two objectives

- **`min_switching`** (the main mode for Analysis B): minimize the **number of open lines**,
  $\sum_{\mathcal{L}^S}(1 - z_{ij})$, with no cost. It answers "what is the smallest $K$ that
  works?" The cut `MIN_OPEN` uses what we already proved ($K \le 3$ infeasible → at least 4), and
  `MAX_OPEN` limits the search to topologies with at most that many openings, so the question
  becomes finite: *is there a topology with 4 to 10 openings?*
- **`cost`**: the usual OTS objective (base-case cost) with a fixed $K$. Used after $K_{\min}$ is
  known, or to test one $K$ at a time.

**Same model as `../ots-sc-dcopf.ipynb`** (same equations, data and parameters). The only
differences are the objective in `min_switching` mode and the subset of states. Section 5 checks
that this notebook reproduces the $K = 0$ cost of the main notebook.

## 0. Settings

Change only this cell. For a long run on the cluster, see the last section (*Running on the
cluster*).

In [ ]:
FORMULATION = "redispatch_only"   # full N-1 set (no exclusions). 'redispatch_with_exclusions' = Analysis A set

MODE     = "min_switching"        # 'min_switching' (find K_min) | 'cost' (fixed K, cost objective)
MIN_OPEN = 4                      # min_switching: at least this many open (K <= 3 is proven infeasible)
MAX_OPEN = 10                     # min_switching: at most this many open (None = no limit)
K_LIST   = [4]                    # cost mode: the K values to test, one after the other

TIME_LIMIT = 3600                  # [s] per core MILP solve
NOREL_TIME = 600                    # [s] Gurobi 'no relaxation' heuristic before the root LP (0 = off)
MIP_GAP    = 1e-2                 # relative gap per core solve (the core result is only a candidate)
MAX_ITER   = 5                  # max constraint-generation iterations
THREADS    = 0                    # Gurobi threads (0 = all cores)

RUN_CHECKS = True                 # Section 5 checks (about 1 min); set False on the cluster

## 1. Data

Same processed data as `../ots-sc-dcopf.ipynb`. This notebook is one folder deeper
(`full-ct-feasibility/`), so the paths start with `../..` (data) and `../` (results).

In [ ]:
import math
import os
import re
import time

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

ROOT   = "../.."
SOURCE = "matpower"
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

# Output: the same folder the old core_screening.py used. ../ots-sc-dcopf.ipynb reads its warm
# start from here (screening/K<k>_topology.csv) when no saved result exists for that K.
OUT_DIR  = f"../results/{SOURCE}/{FORMULATION}/screening"
os.makedirs(OUT_DIR, exist_ok=True)
PROGRESS = f"{OUT_DIR}/progress_{MODE}.log"   # one line per step, to follow a long run

ENV = gp.Env(params={"OutputFlag": 0})   # one Gurobi environment, logs off by default

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")
print(f"output -> {OUT_DIR}/")

## 2. Sets and parameters

Same as `../ots-sc-dcopf.ipynb`: $F^{em} = 1.15 F^{\max}$ in post-contingency states, angle limits
±0.6 rad, per-branch big-M $M_{ij} = (\theta^{\max} - \theta^{\min})/x_{ij}$, switchable set
$\mathcal{L}^S$ = non-radial branches (the radial ones stay closed).

In [ ]:
set_N  = loads["bus"].tolist()
set_G  = gens["gen_id"].tolist()
set_L  = branches["branch_id"].tolist()
set_LS = branches.loc[~branches["is_bridge"], "branch_id"].tolist()   # switchable (non-radial)

par_pd    = dict(zip(loads["bus"],          loads["pd_pu"]))
par_g_bus = dict(zip(gens["gen_id"],        gens["bus"]))
par_pmin  = dict(zip(gens["gen_id"],        gens["pmin_pu"]))
par_pmax  = dict(zip(gens["gen_id"],        gens["pmax_pu"]))
par_cost  = dict(zip(gens["gen_id"],        gens["c1"]))
par_from  = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to    = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x     = dict(zip(branches["branch_id"], branches["x_pu"]))
par_rate  = dict(zip(branches["branch_id"], branches["rate_pu"]))
par_label = dict(zip(branches["branch_id"], branches["label"]))

EMERG_FACTOR = 1.15                                     # F^em = 1.15 F^max
par_rate_em  = {l: EMERG_FACTOR * par_rate[l] for l in set_L}
THETA_MAX    = 0.6                                      # bus angle limits [rad]
THETA_MIN    = -THETA_MAX
par_M        = {l: (THETA_MAX - THETA_MIN) / par_x[l] for l in set_L}   # big-M per branch

set_gens_at   = {i: [] for i in set_N}
set_out_lines = {i: [] for i in set_N}
set_in_lines  = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

print(f"|N| = {len(set_N)} | |G| = {len(set_G)} | |L| = {len(set_L)} | |L^S| = {len(set_LS)}")

## 3. States (scenarios)

Same ids and order as `../ots-sc-dcopf.ipynb`: the base case (id 0), then one state per
non-radial line, then one per generator. `HARD` are the three elements that make the SC-DCOPF
infeasible on the original topology. They are the **initial core** (with the base case), and they
are also the exclusions of Analysis A (`EXCLUDE` in the main notebook).

In [ ]:
HARD = [("line", 7),    # transformer 8-5         (infeasible alone)
        ("line", 95),   # line 38-65              (joint conflict)
        ("gen", 0)]     # generator at bus 10, 505 MW (joint conflict)
EXCLUDE = HARD if FORMULATION == "redispatch_with_exclusions" else []

scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None, "desc": "base case"}]
for l in set_LS:
    if ("line", l) not in EXCLUDE:
        scenarios.append({"id": len(scenarios), "type": "line", "out_line": l, "out_gen": None,
                          "desc": f"line {l} ({par_label[l]}) out"})
for g in set_G:
    if ("gen", g) not in EXCLUDE:
        scenarios.append({"id": len(scenarios), "type": "gen", "out_line": None, "out_gen": g,
                          "desc": f"gen {g} (bus {par_g_bus[g]}) out"})
scen  = {s["id"]: s for s in scenarios}
set_C = list(scen)


def scenario_id(kind, element):
    key = "out_line" if kind == "line" else "out_gen"
    return next(s["id"] for s in scenarios if s["type"] == kind and s[key] == element)


def line_on(l, c):   # N1_{ij,c}
    return 0 if scen[c]["out_line"] == l else 1


def gen_on(g, c):    # N1_{g,c}
    return 0 if scen[c]["out_gen"] == g else 1


def labels(z):
    # labels of the open branches of a topology z = {branch: 0/1}
    return [par_label[l] for l in set_L if z[l] == 0]


INITIAL_CORE = [0] + [scenario_id(k, e) for k, e in HARD if (k, e) not in EXCLUDE]
print(f"FORMULATION = {FORMULATION} | |C| = {len(set_C)} states")
print(f"initial core: {[scen[c]['desc'] for c in INITIAL_CORE]}")

# Losing a generator cannot be fixed by switching if the remaining capacity is below the load.
total_load = sum(par_pd.values())
short = [g for g in set_G if sum(par_pmax.values()) - par_pmax[g] < total_load]
print("capacity check:", "OK (every generator loss leaves enough capacity)" if not short
      else f"WARNING: without gen {short} the capacity is below the load -> infeasible for every K")

## 4. The model, as a function

Unlike `../ots-sc-dcopf.ipynb`, the model is built **many times**: for different subsets of states
(the core grows), with $z$ free (MILP) or fixed (LP), and with two objectives. So it is a function.
The equations are exactly those of the main notebook:

| Block | Equation |
|---|---|
| energy balance | $\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{\Omega_i^+} f_{ij,c} - \sum_{\Omega_i^-} f_{ji,c}$ |
| generator limits | $P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c}$ |
| preventive coupling (line states) | $P_{g,c} = P_{g,0}$ |
| generator response (generator states) | $P_{g,c} = P_{g,0} + \alpha_{g,c} \cdot$ (lost output) |
| thermal limits | $\lvert f_{ij,c} \rvert \le F_{ij} z_{ij}$ ($F^{\max}$ base, $F^{em}$ contingencies) |
| DC flow with big-M | $\lvert f_{ij,c} - (\theta_{i,c} - \theta_{j,c})/x_{ij} \rvert \le M_{ij}(1 - z_{ij})$ |
| line out in state $c$ | $f_{ij,c} = 0$ |
| angles, reference | $\theta^{\min} \le \theta_{i,c} \le \theta^{\max}$, $\theta_{\text{ref},c} = 0$ |

Arguments:
- `states`: the states to include (the base case must be one of them: the dispatch $P_{g,0}$
  couples it to every other state).
- `z_fixed = {branch: 0/1}`: fixes the topology. Then $z$ is a number, the big-M terms are
  constants, and the model is an **LP** (the SC-DCOPF on that topology).
- `objective = "cost"` or `"switching"`; `max_open` / `min_open`: the cuts
  $\sum (1 - z) \le$ `max_open` and $\ge$ `min_open`.

Every constraint name ends in `,c]` (e.g. `fmax[12,49]`), so the state of a constraint can be read
back from its name. Section 6 uses this.

In [ ]:
def build_model(states, z_fixed=None, objective="cost", max_open=None, min_open=0):
    assert 0 in states, "the base case is coupled to every state: it must be included"
    model = gp.Model(env=ENV)

    var_p     = {(g, c): model.addVar(lb=-GRB.INFINITY) for g in set_G for c in states}
    var_f     = {(l, c): model.addVar(lb=-GRB.INFINITY) for l in set_L for c in states}
    var_theta = {(i, c): model.addVar(lb=-GRB.INFINITY) for i in set_N for c in states}

    # --- topology: free binaries (MILP) or fixed numbers (LP) ---
    if z_fixed is None:
        var_z = {l: model.addVar(vtype=GRB.BINARY, name=f"z[{l}]") for l in set_L}
        for l in set_L:
            if l not in set_LS:
                model.addConstr(var_z[l] == 1)              # radial branches stay closed
        n_open = gp.quicksum(1 - var_z[l] for l in set_LS)
        if max_open is not None:
            model.addConstr(n_open <= max_open)             # K (or MAX_OPEN)
        if min_open:
            model.addConstr(n_open >= min_open)             # proven lower bound
    else:
        var_z = z_fixed

    # --- objective ---
    if objective == "switching":
        assert z_fixed is None, "the switching objective needs free z"
        model.setObjective(n_open, GRB.MINIMIZE)
    else:
        model.setObjective(gp.quicksum(par_cost[g] * var_p[(g, 0)] * BASE_MVA for g in set_G),
                           GRB.MINIMIZE)

    # --- constraints, state by state ---
    for c in states:
        kind = scen[c]["type"]
        for i in set_N:                                                       # energy balance
            model.addConstr(gp.quicksum(var_p[(g, c)] for g in set_gens_at[i]) - par_pd[i]
                            == gp.quicksum(var_f[(l, c)] for l in set_out_lines[i])
                            - gp.quicksum(var_f[(l, c)] for l in set_in_lines[i]),
                            name=f"bal[{i},{c}]")
        for g in set_G:                                                       # generator limits
            model.addConstr(var_p[(g, c)] >= par_pmin[g] * gen_on(g, c), name=f"pmin[{g},{c}]")
            model.addConstr(var_p[(g, c)] <= par_pmax[g] * gen_on(g, c), name=f"pmax[{g},{c}]")
        if kind == "line":                                                    # preventive coupling
            for g in set_G:
                model.addConstr(var_p[(g, c)] == var_p[(g, 0)], name=f"couple[{g},{c}]")
        if kind == "gen":                                                     # generator response
            cap_on = sum(gen_on(h, c) * par_pmax[h] for h in set_G)
            lost = gp.quicksum((1 - gen_on(h, c)) * var_p[(h, 0)] for h in set_G)
            for g in set_G:
                if gen_on(g, c):
                    model.addConstr(var_p[(g, c)] == var_p[(g, 0)] + par_pmax[g] / cap_on * lost,
                                    name=f"resp[{g},{c}]")
        rate = par_rate if c == 0 else par_rate_em                            # F^max / F^em
        for l in set_L:
            i, j = par_from[l], par_to[l]
            model.addConstr(var_f[(l, c)] >= -rate[l] * var_z[l], name=f"fmin[{l},{c}]")
            model.addConstr(var_f[(l, c)] <=  rate[l] * var_z[l], name=f"fmax[{l},{c}]")
            if line_on(l, c):                                                 # big-M DC flow
                dc_law = var_f[(l, c)] - (var_theta[(i, c)] - var_theta[(j, c)]) / par_x[l]
                model.addConstr(dc_law + par_M[l] * (1 - var_z[l]) >= 0, name=f"dcflow_lo[{l},{c}]")
                model.addConstr(dc_law - par_M[l] * (1 - var_z[l]) <= 0, name=f"dcflow_hi[{l},{c}]")
            else:                                                             # line out in state c
                model.addConstr(var_f[(l, c)] == 0.0, name=f"dcflow_off[{l},{c}]")
        for i in set_N:                                                       # angle limits
            model.addConstr(var_theta[(i, c)] >= THETA_MIN, name=f"thmin[{i},{c}]")
            model.addConstr(var_theta[(i, c)] <= THETA_MAX, name=f"thmax[{i},{c}]")
        model.addConstr(var_theta[(SLACK_BUS, c)] == 0.0, name=f"ref[{c}]")   # reference bus

    return model, var_z

## 5. Checks: does this notebook reproduce what we already know?

Before trusting a long run, two quick checks (about 1 min), plus the IIS of Section 6 (about 1 min):

1. **Same model as the main notebook.** On the original topology (all $z = 1$) and without the
   three `HARD` states, the cost must be **\$109,681.25/h**, the $K = 0$ result of Analysis A.
2. **The full N-1 set is infeasible on the original topology.** With all states, the same LP must
   be infeasible. This is the starting point of Analysis B.

In [ ]:
ALL_CLOSED = {l: 1 for l in set_L}

if RUN_CHECKS:
    hard_ids = {scenario_id(k, e) for k, e in HARD if (k, e) not in EXCLUDE}

    m, _ = build_model([c for c in set_C if c not in hard_ids], z_fixed=ALL_CLOSED)
    m.optimize()
    print(f"1) all closed, without the HARD states: status {m.status}, "
          f"cost = ${m.ObjVal:,.2f}/h (expected $109,681.25/h)")
    assert m.status == GRB.OPTIMAL and abs(m.ObjVal - 109681.25) < 0.05
    m.dispose()

    if hard_ids:
        m, _ = build_model(set_C, z_fixed=ALL_CLOSED)
        m.setParam("DualReductions", 0)   # report INFEASIBLE (3) instead of INF_OR_UNBD (4)
        m.optimize()
        print(f"2) all closed, full N-1 set: status {m.status} "
              f"({'infeasible, as expected' if m.status == GRB.INFEASIBLE else 'UNEXPECTED'})")
        assert m.status == GRB.INFEASIBLE
        m.dispose()

## 6. Which states conflict with a topology? (IIS)

Given a topology $z^*$ from the core, we solve the **full** problem with $z$ fixed (an LP, all
states together, since they share $P_{g,0}$). If it is infeasible, Gurobi can compute an **IIS**
(*Irreducible Inconsistent Subsystem*): a set of constraints that cannot all hold together, and
that becomes feasible if any one of them is removed. It is the "smallest explanation" of the
infeasibility.

From the names of the IIS constraints (`...,c]`) we read the states involved. Those are the states
the core was missing: they go into the core for the next iteration. The base case is always in
the core already, so it is left out of the list.

An IIS shows **one** conflict at a time, not all of them. That is why the loop may need several
iterations.

In [ ]:
STATE_OF_NAME = re.compile(r",?(\d+)\]$")   # 'fmax[12,49]' -> 49, 'ref[49]' -> 49


def failing_states(z):
    # [] if topology z is feasible for every state, else the contingencies in the IIS
    m, _ = build_model(set_C, z_fixed=z)
    m.setParam("DualReductions", 0)
    m.optimize()
    if m.status == GRB.OPTIMAL:
        m.dispose()
        return []
    m.computeIIS()
    failing = sorted({int(STATE_OF_NAME.search(con.ConstrName).group(1))
                      for con in m.getConstrs() if con.IISConstr} - {0})
    m.dispose()
    return failing


if RUN_CHECKS and EXCLUDE == []:
    t0 = time.time()
    fail = failing_states(ALL_CLOSED)
    print(f"IIS of the original topology ({time.time() - t0:.0f} s): "
          f"{[scen[c]['desc'] for c in fail]}")

## 7. Constraint generation

One function runs the whole loop for one case: `K = None` is the `min_switching` mode (no $K$
limit, objective = number of open lines), an integer `K` is the `cost` mode with that limit.

Each iteration:
1. Solve the **core MILP**. Its Gurobi log goes to `<OUT_DIR>/<tag>_it<n>.log`.
2. If the core is **infeasible** → the full problem is infeasible (proof). Stop.
3. If Gurobi found **no topology** within the time limit → **undecided**. Stop. (Neither a proof
   of infeasibility nor a topology. Try a longer time limit or the cluster.)
4. Otherwise take the topology $z^*$ and call `failing_states(z*)`:
   - no failing state → **$z^*$ is feasible for the full N-1 set**. Save it. Stop.
   - failing states → add them to the core and repeat.

**Bounds in `min_switching` mode.** Each core is a relaxation of the full problem, so the core's
**lower bound** on the number of open lines is also a lower bound for the full problem. If the
final topology opens $n$ lines and the bound is also $n$, then $K_{\min} = n$ is **proven**.
Otherwise we know $\text{bound} \le K_{\min} \le n$.

**Cost mode: feasible is not optimal.** In `cost` mode the cost is minimized **on the core
only**, so the topology found is feasible for the full N-1 set but not necessarily the cheapest
one (`core_cost` is a lower bound, not the full cost). To get the cost, run `../ots-sc-dcopf.ipynb`
for that K: it starts from this topology.

**Outputs** in `OUT_DIR`:
- the topology, as `K<n>_topology.csv` (`n` = open lines in `min_switching` mode, `n` = K in
  `cost` mode; plus `min_switching_topology.csv`). `../ots-sc-dcopf.ipynb` uses
  `K<n>_topology.csv` as its warm start for $K = n$;
- a summary CSV with one row per run;
- `progress_<mode>.log` with one line per step, to follow a long run.

In [ ]:
def log(msg):
    # print and append to the progress file (useful when the notebook runs unattended)
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(line, flush=True)
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def save_topology(z, path):
    pd.DataFrame({"branch_id": set_L, "label": [par_label[l] for l in set_L],
                  "z": [z[l] for l in set_L]}).to_csv(path, index=False)


def screen(K, core):
    core, t0 = list(core), time.time()
    min_sw = K is None
    tag = "minsw" if min_sw else f"K{K}"

    for it in range(1, MAX_ITER + 1):
        # 1) core MILP
        if min_sw:
            model, var_z = build_model(core, objective="switching",
                                       max_open=MAX_OPEN, min_open=MIN_OPEN)
        else:
            model, var_z = build_model(core, objective="cost", max_open=K)
        log_file = f"{OUT_DIR}/{tag}_it{it}.log"
        if os.path.exists(log_file):
            os.remove(log_file)                     # Gurobi appends; keep only this run
        for name, value in [("LogToConsole", 0), ("OutputFlag", 1), ("LogFile", log_file),
                            ("MIPGap", MIP_GAP), ("TimeLimit", TIME_LIMIT), ("MIPFocus", 1),
                            ("DualReductions", 0), ("Threads", THREADS)]:
            model.setParam(name, value)
        if NOREL_TIME:
            model.setParam("NoRelHeurTime", NOREL_TIME)
        model.optimize()
        log(f"{tag} it={it}: core = {len(core)} states, status = {model.status}, "
            f"{model.Runtime:.0f} s")

        row = {"K": "min" if min_sw else K, "iterations": it, "core_size": len(core),
               "core": ";".join(scen[c]["desc"] for c in core)}

        # 2) core infeasible -> full problem infeasible (proof)
        if model.status == GRB.INFEASIBLE:
            if min_sw:
                rng = [f">= {MIN_OPEN}"] * bool(MIN_OPEN) + [f"<= {MAX_OPEN}"] * (MAX_OPEN is not None)
                verdict = "infeasible for every topology" + (f" with {' and '.join(rng)} open" if rng else "")
            else:
                verdict = "infeasible"
            return {**row, "verdict": verdict, "time_s": round(time.time() - t0, 1)}

        # 3) no topology found within the time limit -> undecided
        if model.SolCount == 0:
            return {**row, "verdict": "undecided (time limit, no topology found)",
                    "lower_bound": round(model.ObjBound, 2), "time_s": round(time.time() - t0, 1)}

        z = {l: int(round(var_z[l].X)) for l in set_L}
        core_obj, core_bound = model.ObjVal, model.ObjBound
        model.dispose()
        n_open = sum(1 - z[l] for l in set_LS)

        # 4) full LP on this topology
        failing = failing_states(z)
        log(f"         topology opens {n_open}: {labels(z)}; "
            f"{len(failing)} states in the IIS of the full LP")
        if not failing:
            # warm start for ../ots-sc-dcopf.ipynb: K<n_open> in min_switching mode, K<K> in cost mode
            save_topology(z, f"{OUT_DIR}/K{n_open if min_sw else K}_topology.csv")
            out = {**row, "verdict": "feasible (full N-1)", "n_open": n_open,
                   "opened": ";".join(labels(z)), "time_s": round(time.time() - t0, 1)}
            if min_sw:
                k_lower = max(math.ceil(core_bound - 1e-6), MIN_OPEN)   # integer objective
                save_topology(z, f"{OUT_DIR}/min_switching_topology.csv")
                return {**out, "k_min_lower_bound": k_lower, "k_min_proven": n_open == k_lower}
            return {**out, "core_cost": round(core_obj, 2), "core_bound": round(core_bound, 2)}
        core += [c for c in failing if c not in core]
        log(f"         added to the core: {[scen[c]['desc'] for c in failing]}")

    return {**row, "verdict": f"undecided after {MAX_ITER} iterations",
            "time_s": round(time.time() - t0, 1)}

## 8. Run

This is the long cell. In `min_switching` mode it can take up to `MAX_ITER × TIME_LIMIT` (plus
a few seconds per IIS). The progress is printed and also written to `progress_<mode>.log`.

In [ ]:
log(f"START {MODE} | {FORMULATION} | MIN_OPEN={MIN_OPEN} MAX_OPEN={MAX_OPEN} K_LIST={K_LIST} | "
    f"TIME_LIMIT={TIME_LIMIT} NOREL_TIME={NOREL_TIME} THREADS={THREADS}")

if MODE == "min_switching":
    rows = [screen(None, INITIAL_CORE)]
    summary_file = f"{OUT_DIR}/summary_min_switching.csv"
else:
    rows = []
    for K in K_LIST:
        rows.append(screen(K, INITIAL_CORE))
        log(f"K = {K}: {rows[-1]['verdict']}")
    summary_file = f"{OUT_DIR}/summary_K{'-'.join(map(str, K_LIST))}.csv"

result = pd.DataFrame(rows)
result.to_csv(summary_file, index=False)
log(f"END -> {summary_file}")
display(result.drop(columns="core"))

## 9. Reading the result

| Verdict | Meaning for Analysis B | Next step |
|---|---|---|
| `infeasible for every topology with >= 4 and <= 10 open` | **Proof**: no topology with 4 to 10 openings makes the full N-1 set feasible. | Report it. Optionally raise `MAX_OPEN`. |
| `feasible (full N-1)`, `n_open` = $n$ | A topology with $n$ openings works for every state. $K_{\min} \le n$ (proven equal if `k_min_proven`). | Run `../ots-sc-dcopf.ipynb` with `FORMULATION = "redispatch_only"`, `MAX_SWITCHED = n` (it warm-starts from `screening/K<n>_topology.csv`), then $n+1, \dots$ |
| `undecided ...` | No proof either way within the time limit. | Longer `TIME_LIMIT`, `NOREL_TIME` (e.g. 600), more threads (cluster). |

The core at the end (column `core`) lists the states that mattered: the ones that constrain the
topology. That list is itself informative for the thesis (which contingencies drive the result).

In [ ]:
r = result.iloc[0]
print(f"Verdict: {r['verdict']}")
print(f"Final core ({r['core_size']} states): {r['core'].replace(';', ' | ')}")
if "n_open" in result and pd.notna(r.get("n_open")):
    print(f"Opened ({int(r['n_open'])}): {r['opened'].replace(';', ', ')}")
    if "k_min_proven" in result:
        print("K_min proven" if r["k_min_proven"]
              else f"K_min between {int(r['k_min_lower_bound'])} and {int(r['n_open'])}")

## Running on the cluster

The long run does not need Jupyter open. From `project/ots-sc-dcopf/full-ct-feasibility/`:

1. Set the cell of Section 0 (e.g. `TIME_LIMIT = 7200`, `NOREL_TIME = 600`, `THREADS = 16`,
   `MAX_ITER = 20`, `RUN_CHECKS = False`) and save the notebook.
2. Run it without opening it. The executed copy (with all the outputs) goes to a new file, so the
   original stays clean:
   ```
   jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 ots-full-ct-feasibility.ipynb --output run_minsw.ipynb
   ```
   With `nohup ... &` (Linux) the run survives closing the terminal.
3. Follow the progress with `tail -f ../results/matpower/redispatch_only/screening/progress_min_switching.log`
   (and each Gurobi log, `minsw_it<n>.log`, in the same folder).